# EY Open Science AI & Data Challenge 2026
## Feature Extraction Notebook — Part 1: Earth Engine & Landsat Features

| | |
|---|---|
| **Notebook Purpose** | Extract all remote-sensing and environmental features from external APIs |
| **Outputs** | `training_with_ee_features_enriched_.csv`, `submission_enriched_.csv` |

---

### Overview

This notebook extracts **three families of features** from two remote-sensing pipelines:

| Pipeline | Source API | Features Extracted |
|---|---|---|
| **A — Landsat (Optical)**,  **Terraclimate** | Microsoft Planetary Computer (STAC | `nir`, `green`, `swir16`, `swir22`, `NDMI`, `MNDWI`, `NDVI`, `pet` |
| **B — Static GEE Layers** | Google Earth Engine | `elevation`, `slope`, `aspect`, `soil_ph`, `soil_clay`, `soil_carbon`, `avg_precip`, `land_use`, `veg_index` |
| **C — Dynamic GEE Layers** | Google Earth Engine | `precip_7d_sum`, `precip_30d_sum`, `pet`, `soil_moist_top`, `soil_moist_deep_avg`, `pop_density_3km`, `ag_pct`, `urban_pct`, `forest_pct`, land-cover percentages |

### Execution Order

Run cells **top-to-bottom**. The two pipelines (A and C) are independent and can be run in any order, but both must complete before the preprocessing/modelling notebooks.

```
1. Install dependencies          (Section 1)
2. Authenticate to APIs          (Section 2)
3. Load input data               (Section 3)
4. Pipeline A — Landsat / terraclimate   (Section 4)
5. Pipeline B — Static GEE       (Section 5)
6. Pipeline C — Dynamic GEE      (Section 6)
7. Verify outputs                (Section 7)
```


---
## 1. Install Dependencies

Run this cell once per environment. All libraries are pip-installable.
`earthengine-api` is required for Pipelines B & C; `pystac-client` and `odc-stac` are
required for Pipeline A (Landsat via Planetary Computer).


In [ ]:
# Install all required packages
# Re-run this cell if you encounter ModuleNotFoundError below
%pip install earthengine-api planetary-computer pystac-client odc-stac tqdm geopandas pyproj shapely --quiet


---
## 2. Imports & API Authentication

### 2.1 Standard Imports


In [ ]:
import os
import time
import json
import warnings
import ast
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Planetary Computer / STAC (Pipeline A — Landsat)
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

# Google Earth Engine (Pipelines B & C)
import ee

print('All imports successful.')


### 2.2 Google Earth Engine Authentication

If you are running this notebook for the first time, `ee.Authenticate()` will open a browser
window for OAuth. Once authenticated, the credentials are cached and subsequent calls
only need `ee.Initialize()`.

> **Prerequisite:** your Google Cloud project `ey-challenge-project` must have the
> *Earth Engine API* enabled at https://console.cloud.google.com/apis/library.


In [ ]:
# ── GEE Authentication & Initialization ─────────────────────────────────────
GEE_PROJECT = 'ey-challenge-project'  # ← replace if using a different project

try:
    ee.Initialize(project=GEE_PROJECT)
    print(f'GEE initialized successfully (project={GEE_PROJECT})')
except Exception as e:
    print(f'Authentication required. Opening browser... ({e})')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE initialized successfully after authentication.')


---
## 3. Load Input Data

The two CSV files below are the **original challenge files** (training set and submission
template). All feature extraction pipelines add columns to these files and save enriched
versions to disk.

> Set `DATA_DIR` to the folder containing these CSVs.


In [ ]:
# ── Configuration — update DATA_DIR to your local path ──────────────────────
DATA_DIR = '.'   # folder containing the original challenge CSV files

# Input files
TRAIN_INPUT = os.path.join(DATA_DIR, 'water_quality_training_dataset.csv')
SUB_INPUT   = os.path.join(DATA_DIR, 'submission_template.csv')

# Output files produced by this notebook
TRAIN_OUTPUT = 'training_with_ee_features_enriched_.csv'
SUB_OUTPUT   = 'submission_enriched_.csv'

# ── Load ─────────────────────────────────────────────────────────────────────
train_raw = pd.read_csv(TRAIN_INPUT)
sub_raw   = pd.read_csv(SUB_INPUT)

print(f'Training set:    {train_raw.shape[0]:,} rows x {train_raw.shape[1]} columns')
print(f'Submission set:  {sub_raw.shape[0]:,} rows x {sub_raw.shape[1]} columns')
train_raw.head(3)


---
## 4. Pipeline A — Landsat Optical Features (Microsoft Planetary Computer)

in this part, we use features that were given as starting benchmark.  from the landsat dataset : 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI'. From Terraclimate : pet. 

for more information about how this features have been extracted, two notebooks are included (TerraClimate_Data_Extraction_Notebook.ipynb for **pet** and Landsat_Data_Extraction_Notebook.ipynb for landsat features)


In [ ]:
train_landsat = pd.read_csv('landsat_features_training.csv')
sub_landsat = pd.read_csv('landsat_features_validation.csv')
train_terraclimate = pd.read_csv('terraclimate_features_training.csv')
sub_terraclimate = pd.read_csv('terraclimate_features_validation.csv')

In [ ]:
train_landsat.head()

In [ ]:
train_terraclimate.head()

In [ ]:
#Merge

train_start = pd.merge(
    train_landsat,
    train_terraclimate,
    on=['Longitude', 'Latitude', 'Sample Date'],
    how='inner'
)

sub_start = pd.merge(
    sub_landsat,
    sub_terraclimate,
    on=['Longitude', 'Latitude', 'Sample Date'],
    how='inner'
)


---
## 5. Pipeline B — Static GEE Layers (Terrain, Soil, Land Cover, Vegetation)

### Rationale
These layers are time-independent — they describe long-term environmental conditions
at each measurement station. They are extracted once and joined to all observations
at the same coordinates.

| GEE Dataset | Band(s) Used | Output Feature |
|---|---|---|
| `USGS/SRTMGL1_003` (SRTM DEM) | `elevation`, `slope`, `aspect` | Terrain position & drainage |
| `OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02` | `b0` (surface) | `soil_ph` |
| `OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02` | `b0` | `soil_clay` |
| `OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02` | `b0` | `soil_carbon` |
| `UCSB-CHG/CHIRPS/DAILY` (2011–2015 mean) | `precipitation` | `avg_precip` |
| `ESA/WorldCover/v100` | `Map` | `land_use` (dominant class) |
| `MODIS/061/MOD13Q1` (2011–2015 NDVI mean) | `NDVI` | `veg_index` |


In [ ]:
def extract_static_gee_features(df, chunk_size=1500):
    """
    Extract time-independent (static) GEE features for all unique locations
    in df using a 9-band mega-stack sampled at 30 m.

    The function processes the dataframe in batches to avoid GEE payload
    size limits.  Results are merged back via 'original_index'.

    Parameters
    ----------
    df         : pd.DataFrame  — must contain 'Latitude' and 'Longitude'.
    chunk_size : int           — rows per GEE batch (default 1500).

    Returns
    -------
    pd.DataFrame
        Original dataframe with static GEE columns appended.
    """
    # ── Build the static 9-band image stack ──────────────────────────────────
    dem      = ee.Image('USGS/SRTMGL1_003')
    topo     = ee.Terrain.products(dem).select(['elevation', 'slope', 'aspect'])

    soil_ph  = ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02').select('b0').rename('soil_ph')
    soil_clay= ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_clay')
    soil_c   = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02').select('b0').rename('soil_carbon')
    
    # Long-term average precipitation (2011-2015 baseline)
    avg_rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                  .filterDate('2011-01-01', '2015-12-31')
                  .mean()
                  .rename('avg_precip'))

    land_use = ee.ImageCollection('ESA/WorldCover/v100').first().rename('land_use')

    # Long-term average NDVI (2011-2015 baseline, scale 250 m)
    veg_idx  = (ee.ImageCollection('MODIS/061/MOD13Q1')
                  .filterDate('2011-01-01', '2015-12-31')
                  .select('NDVI')
                  .mean()
                  .rename('veg_index'))

    mega_stack = ee.Image.cat([topo, soil_ph, soil_clay, soil_c, avg_rain, land_use, veg_idx])

    # ── Process in chunks ─────────────────────────────────────────────────────
    df = df.copy()
    chunks   = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]
    results  = []

    print(f'Extracting static features for {len(df)} points in {len(chunks)} batches...')

    for i, chunk in enumerate(tqdm(chunks, desc='Batches')):
        pts = ee.FeatureCollection([
            ee.Feature(
                ee.Geometry.Point([row.Longitude, row.Latitude]),
                {'original_index': row.Index}
            )
            for row in chunk.itertuples()
        ])

        try:
            sampled = mega_stack.sampleRegions(
                collection=pts, scale=30, geometries=False
            ).getInfo()
            batch_df = pd.DataFrame([f['properties'] for f in sampled['features']])
            if not batch_df.empty:
                results.append(batch_df)
        except Exception as e:
            print(f'  Warning — batch {i+1} failed: {e}')

        time.sleep(1)  # respect GEE rate limits

    if not results:
        print('No results extracted. Check GEE authentication.')
        return df

    # ── Merge back via original_index ─────────────────────────────────────────
    all_feats = pd.concat(results, ignore_index=True)
    enriched  = df.merge(all_feats, left_index=True, right_on='original_index', how='left')
    enriched.drop(columns='original_index', inplace=True)
    print(f'Static feature extraction complete. New columns: {list(all_feats.columns)}')
    return enriched


In [ ]:
# ── Run static GEE extraction ─────────────────────────────────────────────────
# NOTE: inputs are the Landsat-enriched DataFrames from Pipeline A (Cell 11),
#       NOT train_raw / sub_raw — otherwise Landsat features would be dropped.
print('=== Training Set — Static GEE Features ===')
train_static = extract_static_gee_features(train_start.copy())
train_static.to_csv('final_training_with_ee_features.csv', index=False)
print(f'Saved: final_training_with_ee_features.csv ({train_static.shape})')

print('\n=== Submission Set — Static GEE Features ===')
sub_static = extract_static_gee_features(sub_start.copy())
sub_static.to_csv('submission_with_environmental_features.csv', index=False)
print(f'Saved: submission_with_environmental_features.csv ({sub_static.shape})')


In [ ]:
tr = pd.read_csv('final_training_with_ee_features.csv')
tr.head()

---
## 6. Pipeline C — Dynamic GEE Features (Time-Aware, Per-Date Extraction)

### Rationale
Unlike the static layers in Section 5, these features are **date-dependent** — they
capture the environmental conditions *at the time of each water sample*.

| Feature | Source | Description |
|---|---|---|
| `precip_7d_sum` | CHIRPS DAILY | Cumulative precipitation over 7 days prior to sampling |
| `precip_30d_sum` | CHIRPS DAILY | Cumulative precipitation over 30 days prior to sampling |
| `precip_30d_max` | CHIRPS DAILY | Maximum daily precipitation in the 30-day window |
| `rainy_days_30d` | CHIRPS DAILY | Number of days with ≥ 1 mm rain in the 30-day window |
| `temp_2m` | ERA5-Land | 2-metre air temperature on sample date |
| `pet` | ERA5-Land | Potential evapotranspiration on sample date |
| `soil_moist_top` | ERA5-Land | Top-layer (0–7 cm) volumetric soil water content |
| `soil_moist_deep_avg` | ERA5-Land | Mean of layers 2–4 (7–289 cm) volumetric soil water |
| `runoff` | ERA5-Land | Surface runoff sum on sample date |
| `evaporation` | ERA5-Land | Total evaporation sum on sample date |
| `pop_density_3km` | WorldPop 2015 | Mean population density within 3 km |
| `evi` | MODIS MOD13A2 | 16-day EVI composite closest to sample date |
| `ndvi` | MODIS MOD13A2 | 16-day NDVI composite closest to sample date |
| `ag_pct` | ESA WorldCover | % agricultural land within 5 km |
| `urban_pct` | ESA WorldCover | % urban land within 5 km |
| `forest_pct` | ESA WorldCover | % forest cover within 5 km |
| `water_pct` | ESA WorldCover | % open water within 5 km |
| `bare_pct` | ESA WorldCover | % bare soil within 5 km |
| `grass_pct` | ESA WorldCover | % grassland within 5 km |

### Processing Strategy
Data is extracted in **configurable chunks** (default: 500 rows for training, 50 for
submission). Chunking prevents GEE payload-size timeouts. Each chunk is converted to a
GEE `FeatureCollection`, a server-side `.map()` call applies `get_point_data()`, and
results are pulled back to the client with `.getInfo()`.


In [ ]:
def extract_dynamic_gee_features(csv_path, output_path, chunk_size=50):
    """
    Extract time-aware (dynamic) GEE features for every row in a CSV file.

    Each row is processed relative to its 'Sample Date', so precipitation
    accumulations, soil moisture and vegetation indices reflect conditions
    at the time the water sample was collected.

    Land-cover histograms (WorldCover) are decoded into percentage columns
    and the raw dictionary is removed to keep the CSV clean.

    Parameters
    ----------
    csv_path    : str  — path to input CSV (must contain Latitude, Longitude, Sample Date).
    output_path : str  — path where the enriched CSV will be saved.
    chunk_size  : int  — rows per GEE batch (500 for training; 50 for submission).

    Returns
    -------
    None  (saves enriched CSV to output_path).
    """
    df = pd.read_csv(csv_path)
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')

    # ── Load GEE datasets (declared once, reused across all chunks) ───────────
    chirps     = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    era5       = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    worldcover = ee.Image('ESA/WorldCover/v100/2020')
    pop        = (ee.ImageCollection('WorldPop/GP/100m/pop')
                    .filter(ee.Filter.eq('year', 2015))
                    .mosaic())
    modis      = ee.ImageCollection('MODIS/061/MOD13A2')
    srtm       = ee.Image('USGS/SRTMGL1_003')

    # ── Server-side per-point feature function ────────────────────────────────
    def get_point_data(feature):
        """
        GEE server-side function: given a point feature with 'date' and geometry,
        return the same feature with all dynamic attributes set.
        """
        date = ee.Date(feature.get('date'))
        geom = feature.geometry()

        # ── Precipitation (CHIRPS) ────────────────────────────────────────────
        p7_sum  = chirps.filterDate(date.advance(-7,  'day'), date).sum()
        p30_sum = chirps.filterDate(date.advance(-30, 'day'), date).sum()
        p30_max = (chirps.filterDate(date.advance(-30, 'day'), date)
                         .max()
                         .reduceRegion(ee.Reducer.mean(), geom, 500)
                         .get('precipitation'))
        rainy_d = (chirps.filterDate(date.advance(-30, 'day'), date)
                         .map(lambda img: img.gte(1))  # binary: rainy day or not
                         .sum()
                         .reduceRegion(ee.Reducer.mean(), geom, 500)
                         .get('precipitation'))

        # ── ERA5-Land (temperature, soil moisture, PET, runoff) ──────────────
        era5_img     = era5.filterDate(date, date.advance(1, 'day')).first()
        temp2m       = (era5_img.select('temperature_2m')
                                .reduceRegion(ee.Reducer.mean(), geom, 500)
                                .get('temperature_2m'))
        soil_top     = (era5_img.select('volumetric_soil_water_layer_1')
                                .reduceRegion(ee.Reducer.mean(), geom, 500)
                                .get('volumetric_soil_water_layer_1'))
        soil_deep    = (era5_img.select(
                            'volumetric_soil_water_layer_2',
                            'volumetric_soil_water_layer_3',
                            'volumetric_soil_water_layer_4'
                        ).reduceRegion(ee.Reducer.mean(), geom, 500).values())  # list[3]
        pet          = (era5_img.select('potential_evaporation_sum')
                                .reduceRegion(ee.Reducer.mean(), geom, 500)
                                .get('potential_evaporation_sum'))
        runoff       = (era5_img.select('runoff_sum')
                                .reduceRegion(ee.Reducer.mean(), geom, 500)
                                .get('runoff_sum'))
        evap         = (era5_img.select('total_evaporation_sum')
                                .reduceRegion(ee.Reducer.mean(), geom, 500)
                                .get('total_evaporation_sum'))

        # ── Land Cover (WorldCover — 5 km buffer histogram) ───────────────────
        lc_hist = worldcover.reduceRegion(
            reducer   = ee.Reducer.frequencyHistogram(),
            geometry  = geom.buffer(5000),
            scale     = 100,
            maxPixels = 1e9
        ).get('Map')   # returned as a dict; decoded in Python below

        # ── Population density (WorldPop — 3 km buffer mean) ─────────────────
        pop_density = pop.reduceRegion(
            ee.Reducer.mean(), geom.buffer(3000), 100
        ).get('population')

        # ── MODIS Vegetation (16-day composite, ±16-day window) ───────────────
        evi_val  = (modis.filterDate(date.advance(-16, 'day'), date)
                        .median().select('EVI')
                        .reduceRegion(ee.Reducer.mean(), geom, 500)
                        .get('EVI'))
        ndvi_val = (modis.filterDate(date.advance(-16, 'day'), date)
                        .median().select('NDVI')
                        .reduceRegion(ee.Reducer.mean(), geom, 500)
                        .get('NDVI'))

        # ── Elevation (SRTM) ──────────────────────────────────────────────────
        elev = srtm.reduceRegion(ee.Reducer.mean(), geom, 90).get('elevation')

        return feature.set({
            'precip_7d_sum'      : p7_sum.reduceRegion(ee.Reducer.mean(), geom, 500).get('precipitation'),
            'precip_30d_sum'     : p30_sum.reduceRegion(ee.Reducer.mean(), geom, 500).get('precipitation'),
            'precip_30d_max'     : p30_max,
            'rainy_days_30d'     : rainy_d,
            'temp_2m'            : temp2m,
            'soil_moist_top'     : soil_top,
            'soil_moist_deep_avg': soil_deep,   # list[3] — averaged in Python
            'runoff'             : runoff,
            'evaporation'        : evap,
            'pop_density_3km'    : pop_density,
            'evi'                : evi_val,
            'ndvi'               : ndvi_val,
            'elevation'          : elev,
            'landcover_hist'     : lc_hist,      # decoded in Python below
            'row_index'          : feature.get('row_index')
        })

    # ── Chunked extraction ────────────────────────────────────────────────────
    results    = []
    total_rows = len(df)
    print(f'Starting dynamic GEE extraction for {total_rows} rows...')

    for start in tqdm(range(0, total_rows, chunk_size), desc='Chunks'):
        end        = min(start + chunk_size, total_rows)
        chunk_df   = df.iloc[start:end].dropna(subset=['Sample Date'])

        if chunk_df.empty:
            continue

        # Build GEE FeatureCollection for this chunk
        features = [
            ee.Feature(ee.Geometry.Point([row['Longitude'], row['Latitude']]), {
                'row_index': idx,
                'date'     : row['Sample Date'].strftime('%Y-%m-%d')
            })
            for idx, row in chunk_df.iterrows()
        ]
        fc = ee.FeatureCollection(features).map(get_point_data)

        try:
            chunk_results = fc.getInfo()['features']

            for f in chunk_results:
                p = f['properties']

                # ── Decode land-cover histogram → percentage columns ───────────
                lc = p.get('landcover_hist', {})
                if isinstance(lc, dict):
                    total_px = sum(lc.values()) or 1
                    # ESA WorldCover class codes (string keys)
                    p['ag_pct']     = (lc.get('40', 0) / total_px) * 100  # cropland
                    p['urban_pct']  = (lc.get('50', 0) / total_px) * 100  # built-up
                    p['forest_pct'] = (lc.get('10', 0) / total_px) * 100  # tree cover
                    p['water_pct']  = (lc.get('80', 0) / total_px) * 100  # open water
                    p['bare_pct']   = (lc.get('60', 0) / total_px) * 100  # bare soil
                    p['grass_pct']  = (lc.get('30', 0) / total_px) * 100  # grassland
                p.pop('landcover_hist', None)  # remove raw dict — not CSV-safe

                # ── Average the three deep soil-moisture layers ───────────────
                deep = p.get('soil_moist_deep_avg')
                if isinstance(deep, list) and len(deep) == 3:
                    p['soil_moist_deep_avg'] = sum(deep) / 3.0
                else:
                    p['soil_moist_deep_avg'] = None

                results.append(p)

        except Exception as e:
            tqdm.write(f'  Warning — chunk {start}:{end} failed: {e}')
            continue

    # ── Merge & save ──────────────────────────────────────────────────────────
    if not results:
        print('No data extracted. Check GEE authentication and dataset availability.')
        return

    enriched_df = pd.DataFrame(results)
    final_df    = df.merge(enriched_df.drop(columns=['date'], errors='ignore'),
                           left_index=True, right_on='row_index')
    final_df.to_csv(output_path, index=False)
    print(f'Saved: {output_path}  ({final_df.shape})')


In [ ]:
# ── Run dynamic GEE extraction ────────────────────────────────────────────────
# Training set: larger chunks are faster (fewer round trips)
extract_dynamic_gee_features(
    csv_path    = 'final_training_with_ee_features.csv',  # output from Section 5
    output_path = TRAIN_OUTPUT,
    chunk_size  = 500
)

# Submission set: smaller chunks improve reliability for the 200-row file
extract_dynamic_gee_features(
    csv_path    = 'submission_with_environmental_features.csv',  # output from Section 5
    output_path = SUB_OUTPUT,
    chunk_size  = 50
)


In [ ]:
trr = pd.read_csv(TRAIN_OUTPUT)
trr.head()

---
## 7. Pipeline D — SoilGrids-ISRIC, MERIT Hydro, TerraClimate & Human Modification

### Rationale
These layers complete the feature set requested by the modelling notebook.

| Feature | Source | Description |
|---|---|---|
| `soil_cec` | SoilGrids-ISRIC `cec_mean` (0–5 cm) | Cation Exchange Capacity — controls nutrient retention |
| `soil_soc` | SoilGrids-ISRIC `soc_mean` (0–5 cm) | Soil Organic Carbon — affects phosphorus cycling |
| `twi` | MERIT Hydro v1.0.1 | Topographic Wetness Index = ln(upstream area / tan(slope)) |
| `human_mod` | CSP Global Human Modification | 0–1 index of human landscape modification |
| `soil_moisture_tc` | TerraClimate `soil` | Monthly soil moisture (±15-day window around sample) |
| `runoff_actual` | TerraClimate `q` | Monthly actual runoff |
| `pdsi` | TerraClimate `pdsi` | Palmer Drought Severity Index |
| `vpd` | TerraClimate `vpd` | Vapour Pressure Deficit |
| `pet_tc` | TerraClimate `pet` | Potential evapotranspiration (TerraClimate) |

### Processing Strategy
Static layers (`soil_cec`, `soil_soc`, `twi`, `human_mod`) are stacked into a
single image and sampled once per unique location (fast).  
TerraClimate temporal layers are queried per-date in chunks of 50 rows.


In [ ]:
def extract_static_extras(df, chunk_size=1500):
    """
    Extracts 5 static features and appends them to df:
        soil_cec   — Cation Exchange Capacity 0-5cm  (SoilGrids-ISRIC)
        soil_soc   — Soil Organic Carbon 0-5cm        (SoilGrids-ISRIC)
        twi        — Topographic Wetness Index         (MERIT Hydro)
        human_mod  — Global Human Modification         (CSP/HM)
        evi_index  — Mean EVI 2011-2015                (MODIS MOD13A2)

    Extraction uses reduceRegion with a fixed 500 m buffer (reproducible).
    GEE is queried once per unique (Latitude, Longitude) pair — not once per row.
    Results are merged back on (Latitude, Longitude) — stable across CSV reloads.
    """
    # ── Build the static image stack ──────────────────────────────────────────
    soil_cec  = ee.Image("projects/soilgrids-isric/cec_mean") \
                  .select("cec_0-5cm_mean").rename("soil_cec")
    soil_soc  = ee.Image("projects/soilgrids-isric/soc_mean") \
                  .select("soc_0-5cm_mean").rename("soil_soc")

    merit     = ee.Image("MERIT/Hydro/v1_0_1")
    slope_rad = ee.Terrain.slope(merit.select("elv")) \
                          .multiply(np.pi / 180).tan().max(0.001)
    twi       = merit.select("upa").divide(slope_rad).log().rename("twi")

    human_mod = ee.ImageCollection("CSP/HM/GlobalHumanModification") \
                  .first().rename("human_mod")

    evi_index = ee.ImageCollection("MODIS/061/MOD13A2") \
                  .filterDate("2011-01-01", "2015-12-31") \
                  .select("EVI").mean().rename("evi_index")

    stack = ee.Image.cat([soil_cec, soil_soc, twi, human_mod, evi_index])
    NEW_COLS = ["soil_cec", "soil_soc", "twi", "human_mod", "evi_index"]

    # ── Deduplicate: query GEE once per unique location, not once per row ─────
    unique_locs = (
        df[["Latitude", "Longitude"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    chunks  = [unique_locs.iloc[i:i + chunk_size]
               for i in range(0, len(unique_locs), chunk_size)]
    records = []

    print(f"Extracting static extras for {len(unique_locs)} unique locations "
          f"({len(df)} total rows) in {len(chunks)} batch(es)...")

    for i, chunk in enumerate(tqdm(chunks, desc="Static extras")):

        # Server-side: map reduceRegion over each point feature
        def sample_point(feature):
            geom = feature.geometry()
            vals = stack.reduceRegion(
                reducer  = ee.Reducer.mean(),
                geometry = geom.buffer(500),   # 500 m buffer — same as Code 2
                scale    = 250,
                maxPixels= 1e9
            )
            return feature.set({
                "soil_cec" : vals.get("soil_cec"),
                "soil_soc" : vals.get("soil_soc"),
                "twi"      : vals.get("twi"),
                "human_mod": vals.get("human_mod"),
                "evi_index": vals.get("evi_index"),
            })

        pts = ee.FeatureCollection([
            ee.Feature(
                ee.Geometry.Point([row.Longitude, row.Latitude]),
                {
                    # String key — survives GEE reordering and CSV round-trips
                    "_loc_key": f"{row.Latitude}_{row.Longitude}"
                }
            )
            for row in chunk.itertuples()
        ]).map(sample_point)

        try:
            sampled = pts.getInfo()["features"]
            for f in sampled:
                p = f["properties"]
                # Reconstruct Lat/Lon from the key so we can merge on them
                lat_str, lon_str = p["_loc_key"].split("_", 1)
                p["Latitude"]  = float(lat_str)
                p["Longitude"] = float(lon_str)
                records.append(p)
        except Exception as e:
            print(f"  Batch {i+1} failed: {e}")

        time.sleep(1)

    if not records:
        print("No data extracted.")
        return df

    # ── Merge on (Latitude, Longitude) — explicit, stable, date-agnostic ──────
    feats_df = (
        pd.DataFrame(records)
          .drop(columns=["_loc_key"], errors="ignore")
          [["Latitude", "Longitude"] + NEW_COLS]
    )

    # Drop any pre-existing versions of the new columns before merging
    df = df.drop(columns=[c for c in NEW_COLS if c in df.columns], errors="ignore")

    enriched = df.merge(feats_df, on=["Latitude", "Longitude"], how="left")

    return enriched

In [ ]:
## — Run & save Pipeline D

# Load the files produced by Pipelines A+B+C (saved by Cell 17).
# These variables may not be in memory if the kernel was restarted between sections.
train_enriched = pd.read_csv(TRAIN_OUTPUT)
sub_enriched   = pd.read_csv(SUB_OUTPUT)

train_enriched = extract_static_extras(train_enriched, chunk_size=1500)
sub_enriched   = extract_static_extras(sub_enriched,   chunk_size=200)

# Save from the in-memory DataFrame, never re-read from an intermediate CSV.
# This guarantees Sample Date and all existing columns remain intact.
train_enriched.to_csv(TRAIN_OUTPUT, index=False)
sub_enriched.to_csv(SUB_OUTPUT,     index=False)

print(f'Training   : {train_enriched.shape}  -> {TRAIN_OUTPUT}')
print(f'Submission : {sub_enriched.shape}    -> {SUB_OUTPUT}')


In [ ]:
trr_ =pd.read_csv('training_with_ee_features_enriched_.csv')
trr_.head()

In [ ]:
trr_.columns

---
## 8. Output Verification

Quick sanity checks to confirm **all** features — from Pipelines A, B, C and D —
were extracted correctly before passing the files to the preprocessing notebook.


In [ ]:
# ── Load the final enriched files ────────────────────────────────────────────
train_enriched = pd.read_csv(TRAIN_OUTPUT)
sub_enriched   = pd.read_csv(SUB_OUTPUT)

print('=== Training Set ===')
print(f'Shape : {train_enriched.shape}')
print(f'Columns ({len(train_enriched.columns)}): {list(train_enriched.columns)}')

print('\n=== Submission Set ===')
print(f'Shape : {sub_enriched.shape}')
print(f'Columns ({len(sub_enriched.columns)}): {list(sub_enriched.columns)}')


In [ ]:
train_enriched.isna().sum()

In [ ]:
sub_enriched.isna().sum()